# Assignment 3: Computer Vision Classification
## Comparing NN, CNN, and Vision Transformer (ViT) on TF Flowers

**Course:** Machine Learning Fundamentals
**Total Marks:** 50
**Due Date:** _(set by lecturer)_

---

## Student Information

**Student Name:** ________________________
**Student ID:** _____________
**Submission Date:** _____________

---

## Assignment Overview

In this assignment you will train and compare **three different families** of image classifiers on the same dataset:

1. A **plain Neural Network** (treats every pixel as an independent number).
2. A **Convolutional Neural Network** (uses convolution + pooling to learn local patterns).
3. A **Vision Transformer** (a pretrained transformer used as a feature extractor).

For each family you will train a **baseline** version and an **optimized** version -- six models in total. At the end you will compare all six head-to-head and reflect on which architecture works best for which job.

### Learning Objectives
- Build and train Dense, Convolutional, and Transformer-based classifiers from scratch.
- Apply common regularisation techniques: dropout, batch normalisation, data augmentation, early stopping.
- Use a pre-trained Hugging Face Vision Transformer as a feature extractor.
- Compare models on the same dataset using accuracy, F1, training time, and parameter count.

### Marks Distribution
| Part | Topic | Marks |
|------|-------|-------|
| 1 | Data exploration & preprocessing | 8 |
| 2 | Neural Network (baseline + optimized) | 12 |
| 3 | Convolutional NN (baseline + optimized) | 12 |
| 4 | Vision Transformer (features + 2 heads) | 10 |
| 5 | Final comparison & reflection | 8 |
| **Total** | | **50** |

---

## Setup and Imports

### Environment
- Use the `mlcourse` conda environment (same as Class Activities and Assignments 1-2).
- Activate: `conda activate mlcourse`
- Kernel: choose **"ML Course (Python 3.10)"** in VS Code.

### Instructions
1. Replace `STUDENT_SEED = XX` with the **last 2 digits of your Student ID**.
2. Run the cells **in order** -- some later cells depend on variables defined earlier.
3. **Do not modify the helper-function cell** (Part 0). You may add cells anywhere else.
4. The Vision Transformer feature extraction (Part 4.1) downloads ~350 MB the first time and takes 5-8 minutes on CPU. Start it early in your session.

> **First run only:** TensorFlow Datasets will download the TF Flowers dataset (~220 MB) into `~/tensorflow_datasets/`. This happens once and is cached for all future runs.

In [ ]:
# === REPLACE XX WITH THE LAST 2 DIGITS OF YOUR STUDENT ID ===
STUDENT_SEED = 42   # e.g. if your ID is 1234567 -> STUDENT_SEED = 67

# === Imports ===
import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# TensorFlow / Keras
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

# Hugging Face transformers + PyTorch (for ViT feature extraction)
import torch
from transformers import AutoImageProcessor, AutoModel
from PIL import Image as PILImage

# === Reproducibility ===
random.seed(STUDENT_SEED)
np.random.seed(STUDENT_SEED)
tf.random.set_seed(STUDENT_SEED)
os.environ['PYTHONHASHSEED'] = str(STUDENT_SEED)

print(f'STUDENT_SEED = {STUDENT_SEED}')
print(f'TensorFlow {tf.__version__}, PyTorch {torch.__version__}')

---

## Helper Functions (Provided)

The cell below defines five helper functions that you will use throughout the assignment. **Do not modify this cell.**

| Function | Purpose |
|----------|---------|
| `load_tf_flowers(image_size, seed)` | Download the TF Flowers dataset, resize, normalise, and split 80/20 (stratified). Returns `X_train, X_test, y_train, y_test, class_names`. |
| `plot_sample_images(X, y, class_names, n)` | Plot `n` random images with class labels. |
| `evaluate_model(model, X_test, y_test, train_time, name)` | Compute accuracy / precision / recall / macro-F1 / param count and return as a dict. |
| `plot_history(history, title)` | Plot training-loss and accuracy curves from a Keras `History` object. |
| `extract_vit_features(images, cache_path)` | Run a pre-trained Vision Transformer on the images and return 768-D feature vectors. Caches to disk. |

In [ ]:
# ===== HELPER FUNCTIONS -- DO NOT MODIFY =====

def load_tf_flowers(image_size=128, seed=42, test_frac=0.2):
    """Load TF Flowers, resize, normalise, stratified 80/20 train/test split."""
    import tensorflow_datasets as tfds
    ds, info = tfds.load('tf_flowers', split='train', with_info=True, as_supervised=True)
    class_names = list(info.features['label'].names)

    def preprocess(img, lbl):
        img = tf.image.resize(img, (image_size, image_size))
        return img / 255.0, lbl

    ds = ds.map(preprocess).batch(64)
    X_batches, y_batches = [], []
    for batch_x, batch_y in tfds.as_numpy(ds):
        X_batches.append(batch_x)
        y_batches.append(batch_y)
    X = np.concatenate(X_batches).astype('float32')
    y = np.concatenate(y_batches)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_frac, stratify=y, random_state=seed)
    return X_train, X_test, y_train, y_test, class_names


def plot_sample_images(X, y, class_names, n=8):
    """Show n random images with their class labels."""
    idx = np.random.choice(len(X), size=n, replace=False)
    cols = min(n, 4); rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
    for ax, i in zip(axes.flat, idx):
        ax.imshow(X[i])
        ax.set_title(class_names[y[i]], fontsize=10)
        ax.axis('off')
    plt.tight_layout(); plt.show()


def evaluate_model(model, X_test, y_test, train_time, name='model'):
    """Compute test metrics. Works for Keras models AND sklearn estimators."""
    if hasattr(model, 'predict_proba') and not hasattr(model, 'predict_classes'):
        # sklearn-style estimator (e.g. LogisticRegression)
        y_pred = model.predict(X_test)
        try:
            params = sum(c.size for c in model.coef_) + model.intercept_.size
        except AttributeError:
            params = 0
    else:
        # Keras model
        y_proba = model.predict(X_test, verbose=0)
        y_pred = y_proba.argmax(axis=1)
        params = model.count_params()

    return {
        'name':       name,
        'accuracy':   accuracy_score(y_test, y_pred),
        'precision':  precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall':     recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_macro':   f1_score(y_test, y_pred, average='macro', zero_division=0),
        'train_time': train_time,
        'params':     params,
        'y_pred':     y_pred,
    }


def plot_history(history, title='Training history'):
    """Plot loss and accuracy curves from a Keras History object."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history.history['loss'], label='train')
    if 'val_loss' in history.history:
        axes[0].plot(history.history['val_loss'], label='val')
    axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
    axes[1].plot(history.history['accuracy'], label='train')
    if 'val_accuracy' in history.history:
        axes[1].plot(history.history['val_accuracy'], label='val')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
    fig.suptitle(title); plt.tight_layout(); plt.show()


def extract_vit_features(images, cache_path=None, batch_size=32, model_name='google/vit-base-patch16-224'):
    """Run pretrained ViT on images (numpy [0,1], any size, RGB). Returns (N, 768) features."""
    if cache_path and os.path.exists(cache_path):
        print(f'  Loading cached features from {cache_path}')
        return np.load(cache_path)

    print(f'  Loading {model_name} (first run downloads ~350 MB)...')
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()

    images_uint8 = (images * 255).astype('uint8')
    n = len(images_uint8)
    feats = []
    for i in range(0, n, batch_size):
        batch = images_uint8[i:i+batch_size]
        pil_batch = [PILImage.fromarray(img).resize((224, 224)) for img in batch]
        inputs = processor(images=pil_batch, return_tensors='pt')
        with torch.no_grad():
            outputs = model(**inputs)
        cls_embeds = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        feats.append(cls_embeds)
        if (i // batch_size) % 5 == 0:
            print(f'  processed {min(i+batch_size, n)}/{n} images')

    feats = np.concatenate(feats)
    if cache_path:
        np.save(cache_path, feats)
        print(f'  Saved features to {cache_path}')
    return feats


print('Helper functions loaded.')

---

# Part 1: Data Exploration and Preprocessing (8 marks)

## 1.1 Load and Explore (4 marks)

**Tasks:**
- Call `load_tf_flowers(image_size=128, seed=STUDENT_SEED)` to get the train/test split.
- Print the shapes of `X_train`, `X_test`, `y_train`, `y_test`.
- Print the class names and the number of images per class in the training set.
- Visualise 8 random training images using `plot_sample_images`.

**Questions:**
- Q1.1.1 -- How many images are in the training set vs the test set?
- Q1.1.2 -- How many classes are there and what are their names?
- Q1.1.3 -- What is the dtype and pixel range of `X_train`? Why is normalisation important here?

In [ ]:
# TODO: load the dataset using load_tf_flowers
# Hint: X_train, X_test, y_train, y_test, class_names = load_tf_flowers(image_size=128, seed=STUDENT_SEED)


# TODO: print the shapes of X_train, X_test, y_train, y_test


# TODO: print the class names and the count per training class
# Hint: use np.unique(y_train, return_counts=True) and zip with class_names


# TODO: visualise 8 random training images using plot_sample_images


**Answer 1.1:**

*Your answers to Q1.1.1, Q1.1.2, Q1.1.3 here. Use the exact numbers from your printed output.*

## 1.2 Verify Shapes and Class Balance (4 marks)

The helper has already done the resize, normalisation, and stratified 80/20 split for you. The optimised models will use `validation_split=0.1` inside `model.fit()` to monitor the validation loss for EarlyStopping -- you do **not** need to make a separate validation set yourself.

**Tasks:**
- Plot a bar chart of the class counts in the training set.
- Plot one image per class with its label (5 images total).

**Questions:**
- Q1.2.1 -- Is the dataset class-balanced? What is the ratio between the largest and smallest class?
- Q1.2.2 -- Mild class imbalance can cause a model to favour the majority class. Why might that NOT be a serious problem in this case?

In [ ]:
# TODO: plot a bar chart of training-set class counts
# Hint:
#   classes, counts = np.unique(y_train, return_counts=True)
#   plt.bar([class_names[c] for c in classes], counts)


# TODO: plot one image per class (5 images total) with its class name as title
# Hint: for each class index c in range(len(class_names)), find np.where(y_train == c)[0][0]


**Answer 1.2:**

*Your answers to Q1.2.1, Q1.2.2 here.*

---

# Part 2: Neural Network (12 marks)

A plain Neural Network treats every pixel as an independent input. It does NOT respect the 2D structure of an image -- this is the baseline against which we will compare the CNN.

## 2.1 Baseline NN (5 marks)

**Tasks:**
- Build a Sequential model with:
  - `Input(shape=(128, 128, 3))`
  - `Flatten()`
  - `Dense(128, activation='relu')`
  - `Dense(5, activation='softmax')`
- Compile with `optimizer='adam'`, `loss='sparse_categorical_crossentropy'`, `metrics=['accuracy']`.
- Fit for **5 epochs**, `batch_size=32`. Time the training using `time.time()`.
- Plot the training history with `plot_history`.
- Evaluate on the test set with `evaluate_model` and store in `nn_base_results`.

**Questions:**
- Q2.1.1 -- What is the test accuracy?
- Q2.1.2 -- Look at the training-accuracy curve. Is the model still improving by epoch 5, or has it plateaued?
- Q2.1.3 -- This NN has more than 6 million parameters but typically reaches only ~65-70% accuracy. Why is that "a lot of parameters for a poor result"?

In [ ]:
# TODO: build the baseline NN
# Hint:
# model_nn_base = models.Sequential([
#     layers.Input(shape=(128, 128, 3)),
#     layers.Flatten(),
#     layers.Dense(128, activation='relu'),
#     layers.Dense(5, activation='softmax'),
# ])


# TODO: compile the model
# Hint: model_nn_base.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


# TODO: train the model for 5 epochs and time the training
# Hint:
# t0 = time.time()
# history_nn_base = model_nn_base.fit(X_train, y_train, epochs=5, batch_size=32, verbose=2)
# train_time = time.time() - t0


# TODO: plot the training history
# Hint: plot_history(history_nn_base, title='NN baseline')


# TODO: evaluate on the test set and store the result
# Hint: nn_base_results = evaluate_model(model_nn_base, X_test, y_test, train_time, name='NN baseline')
#       print(f"test acc: {nn_base_results['accuracy']:.4f}, macro F1: {nn_base_results['f1_macro']:.4f}")


**Answer 2.1:**

*Your answers to Q2.1.1, Q2.1.2, Q2.1.3 here.*

## 2.2 Optimized NN (7 marks)

Now improve the NN with **more capacity + regularisation**. The hope is to keep the same architecture family (no convolutions) but get better generalisation.

**Tasks:**
- Build a Sequential model with these layers in order:
  - `Input(shape=(128, 128, 3))`
  - `Flatten()`
  - `Dense(512, activation='relu')`
  - `BatchNormalization()`
  - `Dropout(0.3)`
  - `Dense(256, activation='relu')`
  - `Dropout(0.3)`
  - `Dense(5, activation='softmax')`
- Compile with the same settings as the baseline.
- Train for up to **15 epochs** with:
  - `validation_split=0.1`
  - `EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)`
- Plot the training history.
- Evaluate on the test set and store in `nn_opt_results`.

**Questions:**
- Q2.2.1 -- What is the test accuracy? How many points did it improve over the baseline?
- Q2.2.2 -- At which epoch did EarlyStopping fire? What was the best `val_loss`?
- Q2.2.3 -- Look at train accuracy vs val accuracy. Is the optimised model overfitting? How can you tell?

In [ ]:
# TODO: build the optimised NN with the layers listed above
# Hint:
# model_nn_opt = models.Sequential([
#     layers.Input(shape=(128, 128, 3)),
#     layers.Flatten(),
#     layers.Dense(512, activation='relu'),
#     layers.BatchNormalization(),
#     layers.Dropout(0.3),
#     layers.Dense(256, activation='relu'),
#     layers.Dropout(0.3),
#     layers.Dense(5, activation='softmax'),
# ])


# TODO: compile


# TODO: define EarlyStopping callback
# Hint: es = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)


# TODO: train with validation_split=0.1 and the callback
# Hint:
# t0 = time.time()
# history_nn_opt = model_nn_opt.fit(X_train, y_train, epochs=15, batch_size=32,
#                                    validation_split=0.1, callbacks=[es], verbose=2)
# train_time = time.time() - t0


# TODO: plot history and evaluate
# Hint:
# plot_history(history_nn_opt, title='NN optimised')
# nn_opt_results = evaluate_model(model_nn_opt, X_test, y_test, train_time, name='NN optimised')
# print(f"test acc: {nn_opt_results['accuracy']:.4f}")


**Answer 2.2:**

*Your answers to Q2.2.1, Q2.2.2, Q2.2.3 here.*

---

# Part 3: Convolutional Neural Network (12 marks)

A CNN respects the 2D structure of the image. It learns local filters and downsamples with pooling.

## 3.1 Baseline CNN (5 marks)

**Tasks:**
- Build a Sequential model with:
  - `Input(shape=(128, 128, 3))`
  - `Conv2D(32, kernel_size=3, activation='relu', padding='same')`
  - `MaxPooling2D(pool_size=2)`
  - `Conv2D(64, kernel_size=3, activation='relu', padding='same')`
  - `MaxPooling2D(pool_size=2)`
  - `Flatten()`
  - `Dense(64, activation='relu')`
  - `Dense(5, activation='softmax')`
- Compile with the standard settings.
- Train for **5 epochs**, `batch_size=32`.
- Plot the training history.
- Evaluate on the test set and store in `cnn_base_results`.

**Questions:**
- Q3.1.1 -- What is the test accuracy? How does it compare to the NN baseline?
- Q3.1.2 -- The CNN has fewer total parameters than the NN baseline (you'll see this in the summary). How can a smaller model be more accurate?
- Q3.1.3 -- Inspect `model_cnn_base.summary()`. After two MaxPooling layers, what is the spatial size before `Flatten`?

In [ ]:
# TODO: build the baseline CNN
# Hint:
# model_cnn_base = models.Sequential([
#     layers.Input(shape=(128, 128, 3)),
#     layers.Conv2D(32, kernel_size=3, activation='relu', padding='same'),
#     layers.MaxPooling2D(pool_size=2),
#     layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'),
#     layers.MaxPooling2D(pool_size=2),
#     layers.Flatten(),
#     layers.Dense(64, activation='relu'),
#     layers.Dense(5, activation='softmax'),
# ])


# TODO: compile, summary, train (5 epochs), plot, evaluate
# Hint: same pattern as Part 2.1
# Save results into: cnn_base_results = evaluate_model(...)


**Answer 3.1:**

*Your answers to Q3.1.1, Q3.1.2, Q3.1.3 here.*

## 3.2 Optimized CNN (7 marks)

Improve the CNN with **deeper architecture + regularisation + data augmentation**.

**Tasks:**
- Build a Sequential model with three Conv blocks (32 -> 64 -> 128 filters), BatchNorm after each Conv, Dropout(0.3) before the final Dense:
  - `Input(shape=(128, 128, 3))`
  - `Conv2D(32, 3, activation='relu', padding='same')` -> `BatchNormalization()` -> `MaxPooling2D(2)`
  - `Conv2D(64, 3, activation='relu', padding='same')` -> `BatchNormalization()` -> `MaxPooling2D(2)`
  - `Conv2D(128, 3, activation='relu', padding='same')` -> `BatchNormalization()` -> `MaxPooling2D(2)`
  - `Flatten()`
  - `Dense(128, activation='relu')` -> `Dropout(0.3)`
  - `Dense(5, activation='softmax')`
- Set up an `ImageDataGenerator` with `rotation_range=20`, `width_shift_range=0.1`, `height_shift_range=0.1`, `horizontal_flip=True`.
- Train for up to **15 epochs** with `batch_size=32` and EarlyStopping.
  - Use `model.fit(datagen.flow(X_train, y_train, batch_size=32), validation_data=(X_test, y_test), ...)`. (For simplicity we use the test set as validation here -- in real research you would carve out a separate val set.)
- Plot the training history.
- Evaluate on the test set and store in `cnn_opt_results`.

**Questions:**
- Q3.2.1 -- What is the test accuracy? How many points did it improve over the baseline CNN?
- Q3.2.2 -- Look at the train vs val accuracy curves. Did augmentation reduce overfitting (train-val gap) compared to the baseline CNN?
- Q3.2.3 -- This optimised CNN has more parameters than the baseline. Compare params vs accuracy gain -- is it a "fair trade"?

In [ ]:
# TODO: build the optimised CNN with three Conv blocks + BatchNorm + Dropout
# Hint:
# model_cnn_opt = models.Sequential([
#     layers.Input(shape=(128, 128, 3)),
#     layers.Conv2D(32, 3, activation='relu', padding='same'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(2),
#     layers.Conv2D(64, 3, activation='relu', padding='same'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(2),
#     layers.Conv2D(128, 3, activation='relu', padding='same'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(2),
#     layers.Flatten(),
#     layers.Dense(128, activation='relu'),
#     layers.Dropout(0.3),
#     layers.Dense(5, activation='softmax'),
# ])


# TODO: compile


# TODO: build the augmentation generator
# Hint:
# datagen = ImageDataGenerator(rotation_range=20, width_shift_range=0.1,
#                              height_shift_range=0.1, horizontal_flip=True)
# datagen.fit(X_train)


# TODO: define EarlyStopping (same as Part 2.2)


# TODO: train with augmentation
# Hint:
# t0 = time.time()
# history_cnn_opt = model_cnn_opt.fit(
#     datagen.flow(X_train, y_train, batch_size=32),
#     epochs=15,
#     validation_data=(X_test, y_test),
#     callbacks=[es],
#     verbose=2,
# )
# train_time = time.time() - t0


# TODO: plot history and evaluate
# Hint:
# plot_history(history_cnn_opt, title='CNN optimised')
# cnn_opt_results = evaluate_model(model_cnn_opt, X_test, y_test, train_time, name='CNN optimised')


**Answer 3.2:**

*Your answers to Q3.2.1, Q3.2.2, Q3.2.3 here.*

---

# Part 4: Vision Transformer (10 marks)

A Vision Transformer slices an image into 16x16 patches and applies self-attention -- no convolution at all. We will not train one from scratch (transformers need millions of images). Instead we use a **pretrained ViT** from Hugging Face as a feature extractor.

## 4.1 Extract ViT Features (Provided -- 2 marks)

The cell below uses `extract_vit_features` to convert each image into a 768-D feature vector. **This step takes 5-8 minutes on CPU the first time** -- after that the features are cached to disk and reload instantly.

**Tasks:**
- Run the cell. Be patient on the first run.
- Print the shapes of `vit_train` and `vit_test`.

**Questions:**
- Q4.1.1 -- What is the dimensionality of one ViT feature vector? (Reminder: it's the output dim of the [CLS] token of `vit-base-patch16-224`.)

In [ ]:
# Extract ViT features for the train and test sets (provided -- do not modify)
print('Extracting ViT features for the training set...')
vit_train = extract_vit_features(X_train, cache_path='vit_features_train.npy')

print('\nExtracting ViT features for the test set...')
vit_test  = extract_vit_features(X_test,  cache_path='vit_features_test.npy')

print(f'\nvit_train shape: {vit_train.shape}')
print(f'vit_test  shape: {vit_test.shape}')

## 4.2 Baseline -- Logistic Regression on ViT Features (4 marks)

Use the simplest possible classifier on top of ViT features.

**Tasks:**
- Fit a `LogisticRegression(max_iter=1000, random_state=STUDENT_SEED)` on `vit_train`, `y_train`.
- Evaluate on `vit_test`, `y_test`. Store in `vit_base_results`.

**Questions:**
- Q4.2.1 -- What is the test accuracy?
- Q4.2.2 -- This linear model has just `768 * 5 + 5 = 3,845` parameters. How does that compare to the optimised CNN's parameter count? Why does it work so well?

In [ ]:
# TODO: fit Logistic Regression on the ViT features
# Hint:
# t0 = time.time()
# clf_vit = LogisticRegression(max_iter=1000, random_state=STUDENT_SEED)
# clf_vit.fit(vit_train, y_train)
# train_time = time.time() - t0


# TODO: evaluate
# Hint: vit_base_results = evaluate_model(clf_vit, vit_test, y_test, train_time, name='ViT + LogReg')
#       print(f"test acc: {vit_base_results['accuracy']:.4f}")


**Answer 4.2:**

*Your answers to Q4.2.1, Q4.2.2 here.*

## 4.3 Optimized -- MLP head on ViT features (4 marks)

Train a small Keras MLP on top of the ViT features. Same features, more capable head.

**Tasks:**
- Build a Sequential model:
  - `Input(shape=(768,))`
  - `Dense(256, activation='relu')`
  - `Dropout(0.3)`
  - `Dense(5, activation='softmax')`
- Compile with the standard settings.
- Train for up to **30 epochs** with `validation_split=0.1` and EarlyStopping (`patience=5`).
- Evaluate on the test set and store in `vit_opt_results`.

**Questions:**
- Q4.3.1 -- What is the test accuracy? How many points did it improve over the LogReg baseline?
- Q4.3.2 -- Training the MLP head is **much faster** than training the CNN even though both classify the same images. Why?

In [ ]:
# TODO: build the small MLP head on ViT features
# Hint:
# model_vit_mlp = models.Sequential([
#     layers.Input(shape=(768,)),
#     layers.Dense(256, activation='relu'),
#     layers.Dropout(0.3),
#     layers.Dense(5, activation='softmax'),
# ])


# TODO: compile, define EarlyStopping with patience=5


# TODO: train and evaluate
# Hint:
# t0 = time.time()
# history_vit_mlp = model_vit_mlp.fit(vit_train, y_train, epochs=30, batch_size=32,
#                                     validation_split=0.1, callbacks=[es], verbose=2)
# train_time = time.time() - t0
# plot_history(history_vit_mlp, title='ViT + MLP')
# vit_opt_results = evaluate_model(model_vit_mlp, vit_test, y_test, train_time, name='ViT + MLP')


**Answer 4.3:**

*Your answers to Q4.3.1, Q4.3.2 here.*

---

# Part 5: Final Comparison and Reflection (8 marks)

## 5.1 Comparison Table (4 marks)

**Tasks:**
- Build a `pandas.DataFrame` with one row per model (6 rows: NN base, NN opt, CNN base, CNN opt, ViT+LogReg, ViT+MLP). Columns: `Family, Variant, Test Accuracy, Macro F1, Train Time (s), Params`.
- Sort by Test Accuracy (descending).
- Plot a grouped bar chart of accuracy by family, with baseline vs optimised side-by-side.

**Questions:**
- Q5.1.1 -- Which is the **single best** model? By how many points does it beat the second-best?
- Q5.1.2 -- Within each family, did optimisation always help? By how many points?

In [ ]:
# TODO: collect all 6 results into a DataFrame
# Hint:
# all_results = [nn_base_results, nn_opt_results,
#                cnn_base_results, cnn_opt_results,
#                vit_base_results, vit_opt_results]
# rows = []
# for r in all_results:
#     family, variant = r['name'].split(' ', 1) if ' ' in r['name'] else (r['name'], '')
#     rows.append({
#         'Family':         family,
#         'Variant':        variant,
#         'Test Accuracy':  round(r['accuracy'], 4),
#         'Macro F1':       round(r['f1_macro'], 4),
#         'Train Time (s)': round(r['train_time'], 1),
#         'Params':         r['params'],
#     })
# summary = pd.DataFrame(rows).sort_values('Test Accuracy', ascending=False)
# display(summary)


# TODO: grouped bar chart of accuracy by family (baseline vs optimised side-by-side)
# Hint: build a small DataFrame indexed by Family with two columns ['baseline', 'optimised']


## 5.2 Confusion Matrices for the Best Model in Each Family (2 marks)

**Tasks:**
- For each family (NN / CNN / ViT) plot the confusion matrix of the **better** of the two variants in that family.
- Use a 1x3 panel of heatmaps with class names as labels.

**Questions:**
- Q5.2.1 -- Which two flower classes are confused most often across all three families?

In [ ]:
# TODO: for each family, pick the better variant and build its confusion matrix
# Hint: choose the variant with higher accuracy in each family
# best_nn  = nn_opt_results  if nn_opt_results['accuracy']  > nn_base_results['accuracy']  else nn_base_results
# best_cnn = cnn_opt_results if cnn_opt_results['accuracy'] > cnn_base_results['accuracy'] else cnn_base_results
# best_vit = vit_opt_results if vit_opt_results['accuracy'] > vit_base_results['accuracy'] else vit_base_results

# TODO: plot 1x3 panel of confusion matrices
# Hint:
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for ax, r in zip(axes, [best_nn, best_cnn, best_vit]):
#     cm = confusion_matrix(y_test, r['y_pred'])
#     sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#                 xticklabels=class_names, yticklabels=class_names, ax=ax, cbar=False)
#     ax.set_title(f"{r['name']}  acc={r['accuracy']:.3f}")
#     ax.set_xlabel('predicted'); ax.set_ylabel('actual')
# plt.tight_layout(); plt.show()


## 5.3 Reflection (2 marks)

Answer all 5 questions below using exact numbers from your own outputs.

- **Q5.3.1** -- By how many percentage points did optimisation improve each family (NN, CNN, ViT)? Which family gained the **most** from optimisation? Briefly explain why.
- **Q5.3.2** -- Which model has the **smallest gap** between train and test accuracy? Why is that property valuable beyond raw accuracy?
- **Q5.3.3** -- The Vision Transformer was pretrained on ImageNet and **never trained on the TF Flowers dataset**. How is it possible that the simple LogReg-on-ViT-features beats the CNN that DID train on this dataset?
- **Q5.3.4** -- Pick one flower class that is consistently confused with another class across all three families (e.g. *daisy* vs *dandelion*). What is the visual reason for that confusion?
- **Q5.3.5** -- If you had to deploy ONE of these models on a low-power phone (limited memory, no GPU), which would you pick and why? Consider accuracy, parameter count, and inference cost.

**Answer 5.3:**

*Your answers here. Refer to specific numbers from your tables.*

---

## Submission Guidelines

### Files to submit
- `<StudentID>_<YourName>_Assignment_3.ipynb` -- this notebook with **all cells executed top to bottom**, with your answers filled in.

### Checklist before you submit
- [ ] Replaced `STUDENT_SEED = XX` with the last 2 digits of your student ID.
- [ ] Filled student name + ID at the top.
- [ ] All TODO cells completed; no `# TODO` comments left.
- [ ] All `*Your answer here...*` placeholders replaced with real answers.
- [ ] Notebook runs end-to-end without errors (Run All).
- [ ] Comparison table in Part 5.1 has all 6 rows.
- [ ] Confusion matrices in Part 5.2 are visible.

### Academic Integrity
- This work must be entirely your own. You may discuss concepts with classmates but **all code, analysis, and written answers must be yours**.
- Do not share notebooks or paste each other's code.
- LLM use: you may use AI tools to **explain concepts**, but you must understand and write all code yourself. Quoting AI text in answer cells is academic misconduct.
- The seed (`STUDENT_SEED`) ensures your results are reproducible. Markers will rerun your notebook -- the same seed should produce the same numbers (within small numerical noise).